# BTCUSDT Microstructure — One-Click Collector v3
**Corrected and hardened.** Metrics use the daily archive path and ISO/epoch-safe timestamp parsing. The notebook stops if OI cannot be parsed.

Run **Runtime → Run all**.

In [ ]:

!pip -q install -U pandas pyarrow requests tqdm python-dateutil
import os,time,json,zipfile,hashlib,requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor,as_completed
import pandas as pd

START=pd.Timestamp(os.environ.get("START_DATE","2025-01-01"),tz="UTC")
END=pd.Timestamp(os.environ.get("END_DATE","2026-05-31"),tz="UTC")
SYMBOL="BTCUSDT"
BASE="https://data.binance.vision/data/futures/um"
ROOT=Path("/content/btcusdt_research_v3")
CACHE,RAW,CAN,REP=[ROOT/x for x in ("cache","raw","canonical","reports")]
for p in (CACHE,RAW,CAN,REP): p.mkdir(parents=True,exist_ok=True)
S=requests.Session()
S.headers.update({"User-Agent":"BTCUSDT-research-collector/3.0"})

def months(a,b):
    x=a.normalize().replace(day=1); out=[]
    while x<=b: out.append(x); x=x+pd.offsets.MonthBegin(1)
    return out

def days(a,b):
    x=a.normalize(); out=[]
    while x<=b: out.append(x); x=x+pd.Timedelta(days=1)
    return out

def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(8*1024*1024),b""): h.update(b)
    return h.hexdigest()

def download(url,target,retries=5):
    target=Path(target); target.parent.mkdir(parents=True,exist_ok=True)
    part=Path(str(target)+".part")
    for attempt in range(retries):
        try:
            n=part.stat().st_size if part.exists() else 0
            r=S.get(url,headers={"Range":f"bytes={n}-"} if n else {},stream=True,timeout=90)
            if r.status_code==404: return "not_found"
            if r.status_code==416 and part.exists():
                part.replace(target); return "ok"
            r.raise_for_status()
            with open(part,"ab" if n else "wb") as f:
                for c in r.iter_content(1024*1024):
                    if c: f.write(c)
            part.replace(target); return "ok"
        except Exception as e:
            print("retry",attempt+1,str(e)); time.sleep(min(2**attempt,20))
    return "failed"

def make_jobs():
    j=[]
    for iv in ("1m","5m"):
        for m in months(START,END):
            s=f"{m.year:04d}-{m.month:02d}"
            j.append(("kline",iv,s,f"{BASE}/monthly/klines/{SYMBOL}/{iv}/{SYMBOL}-{iv}-{s}.zip",CACHE/"klines"/iv/f"{s}.zip"))
    for m in months(START,END):
        s=f"{m.year:04d}-{m.month:02d}"
        j.append(("aggTrades",None,s,f"{BASE}/monthly/aggTrades/{SYMBOL}/{SYMBOL}-aggTrades-{s}.zip",CACHE/"aggTrades"/f"{s}.zip"))
    # CRITICAL: Futures metrics archives are daily.
    for d in days(START,END):
        s=f"{d.year:04d}-{d.month:02d}-{d.day:02d}"
        j.append(("metrics",None,s,f"{BASE}/daily/metrics/{SYMBOL}/{SYMBOL}-metrics-{s}.zip",CACHE/"metrics"/f"{s}.zip"))
    return j

def one(j):
    typ,iv,period,url,path=j
    if path.exists() and path.stat().st_size>100:
        return {"type":typ,"period":period,"url":url,"path":str(path),"status":"cached","sha256":sha256(path)}
    st=download(url,path)
    return {"type":typ,"period":period,"url":url,"path":str(path),"status":st,"sha256":sha256(path) if st=="ok" else None}

jobs=make_jobs()
print("Planned archive objects:",len(jobs))
results=[]
with ThreadPoolExecutor(max_workers=12) as ex:
    fs=[ex.submit(one,j) for j in jobs]
    for i,f in enumerate(as_completed(fs),1):
        rec=f.result(); results.append(rec)
        if i%25==0 or rec["status"]!="ok": print(i,rec["type"],rec["period"],rec["status"])
json.dump(results,open(REP/"download_manifest.json","w"),indent=2)
print("Successful/cached:",sum(r["status"] in ("ok","cached") for r in results),"/",len(results))

for r in results:
    if r["status"] not in ("ok","cached"): continue
    p=Path(r["path"]); dest=RAW/Path(p).relative_to(CACHE).parent
    dest.mkdir(parents=True,exist_ok=True)
    marker=dest/(p.stem+".done")
    if marker.exists(): continue
    try:
        with zipfile.ZipFile(p) as z: z.extractall(dest)
        marker.write_text("ok")
    except Exception as e: print("extract error",p,e)

def parse_metric_time(s):
    # Metrics archives may contain ISO date strings (not epoch ms).
    num=pd.to_numeric(s,errors="coerce")
    out=pd.to_datetime(num,unit="ms",utc=True,errors="coerce")
    mask=out.isna()
    if mask.any(): out.loc[mask]=pd.to_datetime(s.loc[mask],utc=True,errors="coerce")
    return out

# Klines
kparts=[]
for p in (RAW/"klines").rglob("*.csv"):
    try:
        d=pd.read_csv(p,header=None,low_memory=False)
        if d.shape[1]<12: continue
        d=d.iloc[:,:12].copy()
        d.columns=["open_ms","open","high","low","close","volume","close_ms","quote_volume","trade_count","taker_buy_base","taker_buy_quote","ignore"]
        d["ts"]=pd.to_datetime(pd.to_numeric(d.open_ms,errors="coerce"),unit="ms",utc=True)
        for c in ["open","high","low","close","volume","quote_volume","taker_buy_base","taker_buy_quote","trade_count"]: d[c]=pd.to_numeric(d[c],errors="coerce")
        kparts.append(d[["ts","open","high","low","close","volume","quote_volume","trade_count","taker_buy_base","taker_buy_quote"]])
    except Exception as e: print("kline parse error",p,e)
if not kparts: raise RuntimeError("FATAL: no usable kline archives.")
k=pd.concat(kparts,ignore_index=True).drop_duplicates("ts").sort_values("ts")
k=k[(k.ts>=START)&(k.ts<=END)].copy()
k["delta_taker_base"]=2*k.taker_buy_base-k.volume
k["ret_5m"]=k.close.pct_change()
k["flow_z_4h"]=(k.delta_taker_base-k.delta_taker_base.rolling(48).mean())/k.delta_taker_base.rolling(48).std()
k["rv_4h"]=k.ret_5m.rolling(48).std()

# Metrics / OI / crowding
mparts=[]
for p in (RAW/"metrics").rglob("*.csv"):
    try:
        d=pd.read_csv(p,low_memory=False); d.columns=[str(c).strip() for c in d.columns]
        tc=next((c for c in d.columns if c.lower()=="create_time"),None)
        if tc is None: continue
        d["ts"]=parse_metric_time(d[tc])
        ren={}
        for c in d.columns:
            q=str(c).lower()
            if q=="sum_open_interest": ren[c]="oi"
            elif "count_toptrader_long_short_ratio" in q: ren[c]="top_ls_count"
            elif "sum_toptrader_long_short_ratio" in q: ren[c]="top_ls_position"
            elif "count_long_short_ratio" in q: ren[c]="global_ls_count"
            elif "sum_taker_long_short_vol_ratio" in q: ren[c]="taker_ls"
        d=d.rename(columns=ren)
        keep=["ts"]+[c for c in ["oi","top_ls_count","top_ls_position","global_ls_count","taker_ls"] if c in d.columns]
        qd=d[keep].dropna(subset=["ts"])
        if len(qd): mparts.append(qd)
    except Exception as e: print("metrics parse error",p,e)
if not mparts: raise RuntimeError("FATAL: metrics archives were downloaded but no metric rows were parsed.")
m=pd.concat(mparts,ignore_index=True).drop_duplicates("ts").sort_values("ts")
m=m[(m.ts>=START)&(m.ts<=END)]
k=k.merge(m,on="ts",how="left")
if "oi" not in k.columns or k["oi"].notna().sum()==0: raise RuntimeError("FATAL: OI is empty after join. Stopping to prevent a false OI-free dataset.")
k["oi_change_5m"]=k.oi.diff()
k["oi_change_1h"]=k.oi.diff(12)
k["oi_change_4h"]=k.oi.diff(48)
k["oi_pct_change_1h"]=k.oi.pct_change(12)
if "taker_ls" in k.columns: k["crowding_z_4h"]=(k.taker_ls-k.taker_ls.rolling(48).mean())/k.taker_ls.rolling(48).std()

# AggTrades
aparts=[]
for p in (RAW/"aggTrades").rglob("*.csv"):
    try:
        d=pd.read_csv(p,header=None,low_memory=False)
        if d.shape[1]<7: continue
        d=d.iloc[:,:7].copy(); d.columns=["id","price","qty","first_id","last_id","ms","buyer_maker"]
        d["ts"]=pd.to_datetime(pd.to_numeric(d.ms,errors="coerce"),unit="ms",utc=True)
        d["qty"]=pd.to_numeric(d.qty,errors="coerce")
        d["signed_qty"]=d.qty.where(~d.buyer_maker,-d.qty)
        d["bucket"]=d.ts.dt.floor("5min")
        aparts.append(d[["id","qty","signed_qty","bucket"]])
    except Exception as e: print("agg parse error",p,e)
if aparts:
    a=pd.concat(aparts,ignore_index=True).drop_duplicates("id")
    af=a.groupby("bucket").agg(agg_delta=("signed_qty","sum"),agg_trades=("id","size"),avg_trade_size=("qty","mean")).reset_index().rename(columns={"bucket":"ts"})
    k=k.merge(af,on="ts",how="left")

# Forward targets (not predictors)
k["fwd_ret_5m"]=k.close.shift(-1)/k.close-1
k["fwd_ret_15m"]=k.close.shift(-3)/k.close-1
k["fwd_ret_60m"]=k.close.shift(-12)/k.close-1
k=k.sort_values("ts").reset_index(drop=True)
k.to_parquet(CAN/"btcusdt_futures_5m.parquet",index=False,compression="zstd")

u=k.ts.drop_duplicates(); diff=u.diff().dt.total_seconds().div(60)
g=pd.DataFrame({"from_ts":u.iloc[:-1].to_numpy(),"to_ts":u.iloc[1:].to_numpy(),"gap_minutes":diff.iloc[1:].to_numpy()}); g=g[g.gap_minutes>5]
g.to_csv(REP/"gaps.csv",index=False)
quality={"rows":int(len(k)),"start":str(k.ts.min()),"end":str(k.ts.max()),"duplicate_timestamps":int(k.ts.duplicated().sum()),"gap_count_gt_5m":int(len(g)),"null_close":int(k.close.isna().sum()),"oi_non_null":int(k.oi.notna().sum()),"crowding_non_null":int(k.taker_ls.notna().sum()) if "taker_ls" in k else 0,"agg_flow_buckets":int(k.agg_delta.notna().sum()) if "agg_delta" in k else 0}
json.dump(quality,open(REP/"quality.json","w"),indent=2)
print("FINAL QUALITY",json.dumps(quality,indent=2))
print("CANONICAL",CAN/"btcusdt_futures_5m.parquet")

pkg=Path("/content/btcusdt_microstructure_research_package.zip")
with zipfile.ZipFile(pkg,"w",zipfile.ZIP_DEFLATED) as z:
    for p in ROOT.rglob("*"):
        if p.is_file(): z.write(p,p.relative_to(ROOT))
print("PACKAGE",pkg,"MB",round(pkg.stat().st_size/1024/1024,2))
from google.colab import files
files.download(str(pkg))
